## Converting the TDMS to CSV files 

TMA high-resolution telemetry is stored in the TMA MCS computer as TDMS files compressed as ZIP files. With this notebook you can convert TDMS to CSV files. 

The TDMS is a specific format that requires a special package to allow reading these files.
Use the cell below to install it locally since Rubin Science Pipelines does not have it installed by default. 

In [ ]:
# You will need this package since it does not come with RSP
!pip install --quiet nptdms 

In [ ]:
import pandas as pd
from nptdms import TdmsFile

In [ ]:
def tdms_to_csv(tdms_path: str, csv_path: str):
    """
    Convert a TDMS file to a CSV file.
    
    Parameters:
    tdms_path (str): Path to the input TDMS file.
    csv_path (str): Path to save the output CSV file.
    """
    tdms_file = TdmsFile.read(tdms_path)
    
    data = {}
    max_length = 0
    
    for group in tdms_file.groups():
        for channel in group.channels():
            data[channel.name] = channel[:]
            max_length = max(max_length, len(channel[:]))
    
    for key in data:
        if len(data[key]) < max_length:
            data[key] = list(data[key]) + [None] * (max_length - len(data[key]))
    
    df = pd.DataFrame(data)
    
    df.to_csv(csv_path, index=False)
    print(f"CSV file saved to {csv_path}")

Here is one example on how to use the function. The file in this example is connected to the Telemetry data taken from the LSSTCam computer, for the rotator angle and CCW torque values. 

In [ ]:
tdms_to_csv("TelemetryData_2025_02_18_14_40.tdms", "TelemetryData_2025_02_18_14_40.csv")